# Video prep on Colab: cut, transcribe, background swap → ready for Claude Code

Works for **vertical (9:16, Reels/TikTok)** and **horizontal (16:9, YouTube)** videos — detected automatically.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.
In Google Drive: `My Drive/video-editor/raw footage/` (put your clip here) and `My Drive/video-editor/results/`
(share it once as *Anyone with the link*).

**Run order**
1. **Step 0** (settings) → **Part A** (≈ 12–18 min for a 5-min recording). Part A cuts silences + retakes, transcribes
   (Darija + English/French), fixes loudness, and saves everything to `results/<slug>/`.
   - `CHANGE_BACKGROUND` off → you're done: the video is graded and ready. Give Claude the folder link.
   - `CHANGE_BACKGROUND` on → Part A also saves `frame_for_image_ai.png`. Give it to your image AI, save the result as
     `results/<slug>/background.png`, then run **Part B** (all its cells, top to bottom).
2. **Part B** cuts you out of every frame (MatAnyone 2), puts you in the new room, applies the soft grade and
   replaces `results/<slug>/edit.mp4`. The original is kept as `edit_original_background.mp4`.
3. Give Claude Code the link of `results/<slug>/`.

To re-tune the look (colour match, light wrap, blur, grain): change the sliders in Step 0, run Step 0, then run only
**B5 and B6** again (the cut-out is kept).

Models: Darija = `anaszil/whisper-large-v3-turbo-darija` · English/French = `openai/whisper-large-v3-turbo` ·
cut-out = MatAnyone 2 (non-commercial S-Lab licence) + BiRefNet-portrait · background fill = LaMa.

In [ ]:
#@title Step 0 · Settings
#@markdown **Name of this video** (short, no spaces) — becomes the folder `results/<slug>/`.
SLUG = "my-video"  #@param {type:"string"}
#@markdown Clip in `video-editor/raw footage/`. Empty = the newest file there.
VIDEO_NAME = ""  #@param {type:"string"}
ROOT = "/content/drive/MyDrive/video-editor"  #@param {type:"string"}
#@markdown Only detect retakes (list them) but don't cut them.
KEEP_RETAKES = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Background swap** (Part B). Tick it BEFORE running Part A (then Part A leaves the grade for later).
CHANGE_BACKGROUND = True  #@param {type:"boolean"}
#@markdown Your AI background image. Empty = `results/<slug>/background.png`.
BACKGROUND_IMAGE = ""  #@param {type:"string"}
#@markdown Tick if you are still in the image (made from your frame). Untick for an empty room.
BACKGROUND_HAS_ME = True  #@param {type:"boolean"}
#@markdown Cut-out quality: 1080 = best edges (slower), 720 = about twice as fast.
MATTING_SIZE = 1080  #@param [720, 1080] {type:"raw"}

#@markdown ---
#@markdown **Realism** (change, run Step 0, then B5 + B6 again).
COLOR_MATCH = 0.25  #@param {type:"slider", min:0, max:1, step:0.05}
LIGHT_WRAP = 0.35  #@param {type:"slider", min:0, max:1, step:0.05}
BACKGROUND_BLUR = 1.5  #@param {type:"slider", min:0, max:8, step:0.5}
GRAIN = 1.0  #@param {type:"slider", min:0, max:2, step:0.1}
EDGE_TIGHTEN = 0.0  #@param {type:"slider", min:0, max:0.3, step:0.02}

import os
RES = f"{ROOT}/results/{SLUG}"
if not BACKGROUND_IMAGE:
    BACKGROUND_IMAGE = f"{RES}/background.png"
OUTPUT_SIZE = "1080p"   # long side 1920 max (vertical 1080x1920, horizontal 1920x1080)
print("Settings saved. Results folder:", RES)

In [ ]:
#@title Part A · Prep: convert, silence cut, transcription (Darija + EN/FR), retakes, loudness (+ grade)
import subprocess, time, json, os
from IPython.display import Image as IPImage, display
t0 = time.time()
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU → Save, then run Step 0 and this cell again.")
print("GPU:", gpu)
from google.colab import drive
drive.mount("/content/drive")
!wget -q -O /content/colab_prep.py https://raw.githubusercontent.com/anassajrx/video-editorV/claude/hopeful-clarke-qhnr9z/local/colab/colab_prep.py
flags = f'--slug "{SLUG}" --root "{ROOT}"'
if VIDEO_NAME: flags += f' --video "{VIDEO_NAME}"'
if KEEP_RETAKES: flags += " --no-retakes"
if CHANGE_BACKGROUND: flags += " --no-grade"      # the grade is applied after the swap (Part B)
!cd /content && python colab_prep.py {flags}
assert os.path.exists(f"{RES}/edit.mp4"), "Part A did not finish: read the error above (or send it to Claude)."
meta = json.load(open(f"{RES}/meta.json"))
print(f"\nPart A done in {(time.time() - t0) / 60:.1f} min · {meta['width']}x{meta['height']} {meta['orientation']} · "
      f"{meta['finalDurationS']:.0f} s after cuts")
if CHANGE_BACKGROUND:
    frame = f"{RES}/frame_for_image_ai.png"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", "1", "-i", f"{RES}/edit.mp4", "-frames:v", "1", frame], check=True)
    display(IPImage(frame, width=260 if meta["orientation"] == "vertical" else 420))
    print(f"""
Background swap — your turn:
1. Download {frame} and give it to your image AI with:
   "Keep the person exactly the same. Replace only the background with [your room]. Same camera angle, same framing,
    same image size ({meta['width']}x{meta['height']}). Photorealistic, natural light coming from the same side as the light on my face."
2. Save the result to Drive as {RES}/background.png
3. Run Part B (B1 → B6).""")
else:
    print("No background swap: the folder is ready. Give Claude Code the link of", RES)

## Part B · Background swap (only if `CHANGE_BACKGROUND` is on)
Run after you saved your AI image as `results/<slug>/background.png`. B4b (the cut-out) is the slow step: it prints
frames/s and the time left. For a 5-min recording (~3.5 min after cuts, ~6,300 frames) expect roughly 30–45 min for all of Part B at 1080, ~20–30
at 720.

In [ ]:
#@title B1 · Setup for the background swap (about 1–2 minutes)
import os, sys, json, time, math, subprocess, shutil
from pathlib import Path
T_SETUP = time.time()

def run(cmd, quiet=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("Command failed: " + " ".join(map(str, cmd[:8])) + " ...")
    return r.stdout

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU found. Go to Runtime → Change runtime type → T4 GPU, click Save, then run this cell again.")
print("GPU:", gpu)

from google.colab import drive
drive.mount("/content/drive")
assert CHANGE_BACKGROUND, "CHANGE_BACKGROUND is off in Step 0: Part B is not needed."

MA2_DIR = Path("/content/MatAnyone2")
MA2_COMMIT = "0079197acd6d16a741f71558809c06c586c579e0"   # the version this notebook was tested with
if not MA2_DIR.exists():
    run(["git", "clone", "-q", "https://github.com/pq-yang/MatAnyone2", str(MA2_DIR)])
    run(["git", "-C", str(MA2_DIR), "checkout", "-q", MA2_COMMIT])
sys.path.insert(0, str(MA2_DIR))

# MatAnyone 2's own package install is broken with new build tools, so we use the code directly
# and only add what Colab doesn't already have. --no-deps keeps Colab's own versions untouched.
!pip -q install hydra-core omegaconf onnxruntime pooch pymatting
!pip -q install --no-deps rembg simple-lama-inpainting

import numpy as np, torch, torch.nn.functional as F
from PIL import Image
from IPython.display import display, HTML
DEV = torch.device("cuda")
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
print(f"Setup done in {time.time() - T_SETUP:.0f} s.")

In [ ]:
#@title B2 · Helpers (just run this cell)
def probe(path):
    out = run(["ffprobe", "-v", "error", "-show_entries",
               "stream=codec_type,width,height,avg_frame_rate,r_frame_rate,color_transfer,pix_fmt:"
               "stream_side_data=rotation:format=duration", "-of", "json", str(path)])
    j = json.loads(out)
    v = next(s for s in j["streams"] if s["codec_type"] == "video")
    rate = v.get("avg_frame_rate") or v.get("r_frame_rate") or "30/1"
    num, den = (rate.split("/") + ["1"])[:2]
    fps = float(num) / float(den) if float(den) else 30.0
    return dict(w=int(v["width"]), h=int(v["height"]), fps=fps,
                dur=float(j["format"].get("duration", 0)),
                audio=any(s["codec_type"] == "audio" for s in j["streams"]),
                transfer=v.get("color_transfer", ""), pix_fmt=v.get("pix_fmt", ""))

def to_seconds(t):
    s = 0.0
    for p in str(t).strip().split(":"):
        s = s * 60 + float(p)
    return s

READ_VF = "scale=in_color_matrix=bt709:in_range=tv"            # how we read our own clip
WRITE_VF = "scale=out_color_matrix=bt709:out_range=tv,format=yuv420p"
TAGS = ["-colorspace", "bt709", "-color_primaries", "bt709", "-color_trc", "bt709"]
X264 = ["-c:v", "libx264", "-preset", "veryfast", "-crf", "16", "-pix_fmt", "yuv420p"]

def read_frames(path, w, h, gray=False):
    """Yield frames one at a time (never the whole video in memory)."""
    ch = 1 if gray else 3
    cmd = ["ffmpeg", "-v", "error", "-i", str(path)]
    if not gray:
        cmd += ["-vf", READ_VF]
    cmd += ["-f", "rawvideo", "-pix_fmt", "gray" if gray else "rgb24", "-"]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=w * h * ch * 4)
    n = w * h * ch
    try:
        while True:
            b = p.stdout.read(n)
            if len(b) < n:
                break
            yield np.frombuffer(b, np.uint8).reshape(h, w, ch)
    finally:
        p.stdout.close(); p.wait()

def writer(path, w, h, fps, gray=False, audio_from=None):
    cmd = ["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "gray" if gray else "rgb24",
           "-s", f"{w}x{h}", "-r", f"{fps:.6f}", "-i", "-"]
    if gray:
        cmd += ["-c:v", "ffv1", str(path)]          # lossless cut-out mask
    else:
        if audio_from:
            cmd += ["-i", str(audio_from), "-map", "0:v", "-map", "1:a?", "-c:a", "aac", "-b:a", "192k", "-shortest"]
        cmd += ["-vf", WRITE_VF, *X264, *TAGS, "-movflags", "+faststart", str(path)]
    return subprocess.Popen(cmd, stdin=subprocess.PIPE)

def gblur(x, sigma):
    """Fast separable Gaussian blur on a (C,H,W) GPU tensor."""
    if sigma <= 0.05:
        return x
    r = max(1, int(math.ceil(sigma * 3)))
    k = torch.arange(-r, r + 1, device=x.device, dtype=x.dtype)
    k = torch.exp(-(k * k) / (2 * sigma * sigma)); k = k / k.sum()
    c = x.shape[0]
    y = F.pad(x[None], (r, r, 0, 0), mode="replicate")
    y = F.conv2d(y, k.view(1, 1, 1, -1).repeat(c, 1, 1, 1), groups=c)
    y = F.pad(y, (0, 0, r, r), mode="replicate")
    y = F.conv2d(y, k.view(1, 1, -1, 1).repeat(c, 1, 1, 1), groups=c)
    return y[0]

def big_blur(x, sigma):
    """Large blurs done at a quarter size (much faster, looks the same)."""
    if sigma < 6:
        return gblur(x, sigma)
    h, w = x.shape[-2:]
    s = F.interpolate(x[None], scale_factor=0.25, mode="area")[0]
    s = gblur(s, sigma / 4)
    return F.interpolate(s[None], size=(h, w), mode="bilinear", align_corners=False)[0]

_SEG = {}   # kept for compatibility (the person-finder now runs in its own process)
_PM_SCRIPT = """
import sys, numpy as np
from PIL import Image
from rembg import new_session, remove
s = new_session(sys.argv[3])
m = remove(Image.open(sys.argv[1]).convert("RGB"), session=s, only_mask=True)
np.save(sys.argv[2], np.array(m.convert("L")))
"""
def person_mask(img_rgb):
    """Soft mask of the person in one image (0..255), BiRefNet-portrait through rembg.
    Runs in a separate process: BiRefNet peaks at ~9.5 GB RAM on CPU, and the memory is only fully
    released when that process exits (Colab has ~12.7 GB)."""
    pth = WORK / "_pm_in.png"; outp = WORK / "_pm_out.npy"; scr = WORK / "_pm.py"
    scr.write_text(_PM_SCRIPT)
    Image.fromarray(img_rgb).save(pth)
    print("Finding you in the image (BiRefNet-portrait; first time downloads ~1 GB)...")
    r = subprocess.run([sys.executable, str(scr), str(pth), str(outp), SEG_MODEL], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("The person-finder failed (see the error above).")
    m = np.load(outp)
    return np.array(Image.fromarray(m).resize((img_rgb.shape[1], img_rgb.shape[0]), Image.BILINEAR))

SEG_MODEL = "birefnet-portrait"

def show_images(paths_or_arrays, width=220):
    from io import BytesIO
    import base64
    tags = []
    for item in paths_or_arrays:
        im = Image.open(item) if isinstance(item, (str, Path)) else Image.fromarray(item)
        im = im.convert("RGB"); im.thumbnail((width * 2, width * 4))
        buf = BytesIO(); im.save(buf, format="JPEG", quality=85)
        tags.append(f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}" '
                    f'style="width:{width}px;margin-right:8px;border-radius:6px">')
    display(HTML("".join(tags)))

print("Helpers loaded.")

In [ ]:
#@title B3 · Load the cut video from Part A
t0 = time.time()
src = Path(RES) / "edit.mp4"
assert src.exists(), f"Can't find {src}. Run Part A first."
OUT = Path(RES) / "background_files"; OUT.mkdir(exist_ok=True)
info = probe(src)
fps = info["fps"] if 1 < info["fps"] <= 120 else 30.0
fps = round(fps * 1000) / 1000
segment = WORK / "segment.mp4"
# whole cut video, BT.709-tagged (same frames, same timing as edit.mp4), long side max 1920 for both orientations
vf = ["scale='if(gt(iw,ih),min(1920,iw),-2)':'if(gt(iw,ih),-2,min(1920,ih))':flags=lanczos",
      "scale=out_color_matrix=bt709:out_range=tv,format=yuv420p"]
run(["ffmpeg", "-v", "error", "-y", "-i", str(src), "-vf", ",".join(vf), "-vsync", "cfr", "-r", f"{fps}", *X264, *TAGS,
     "-c:a", "aac", "-b:a", "192k", str(segment)])
seg = probe(segment)
W, H, FPS = seg["w"], seg["h"], seg["fps"]
N_FRAMES = int(round(seg["dur"] * FPS))
frame0 = next(read_frames(segment, W, H))
print(f"{W}x{H} ({'vertical' if H >= W else 'horizontal'}), {FPS:.2f} fps, {seg['dur']:.1f} s, {N_FRAMES} frames "
      f"({time.time() - t0:.0f} s)")

In [ ]:
#@title B4 · Prepare the background (removes you from the image if needed)
t0 = time.time()
bgp = Path(BACKGROUND_IMAGE)
assert bgp.exists(), f"Can't find {bgp}. Check BACKGROUND_IMAGE in Step 0."
bg = Image.open(bgp).convert("RGB")

# Fit the image to the video: fill the frame, crop the overflow from the centre.
ar_img, ar_vid = bg.width / bg.height, W / H
if abs(ar_img - ar_vid) / ar_vid > 0.03:
    print(f"Note: your image is {bg.width}x{bg.height} and the video is {W}x{H}. "
          "The image will be cropped to fit. For the best result, make it the same shape as the video.")
scale = max(W / bg.width, H / bg.height)
bg = bg.resize((math.ceil(bg.width * scale), math.ceil(bg.height * scale)), Image.LANCZOS)
left, top = (bg.width - W) // 2, (bg.height - H) // 2
bg = np.array(bg.crop((left, top, left + W, top + H)))

if BACKGROUND_HAS_ME:
    print("Removing you from the background image...")
    import cv2
    pm = person_mask(bg) > 40
    k = max(9, int(0.03 * min(W, H)) | 1)            # grow the hole a bit so no edge of you is left
    hole = cv2.dilate(pm.astype(np.uint8) * 255, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    # LaMa works best around 1–1.3 MP, so fill at that size and paste only the hole back.
    f = min(1.0, 1280 / max(W, H))
    sw, sh = int(W * f) // 8 * 8, int(H * f) // 8 * 8
    from simple_lama_inpainting import SimpleLama
    lama = SimpleLama()
    filled = lama(Image.fromarray(bg).resize((sw, sh), Image.LANCZOS),
                  Image.fromarray(hole).resize((sw, sh), Image.NEAREST))
    filled = np.array(filled.crop((0, 0, sw, sh)).resize((W, H), Image.LANCZOS)).astype(np.float32)
    soft = cv2.GaussianBlur(hole, (0, 0), k / 6).astype(np.float32)[..., None] / 255
    bg = (bg * (1 - soft) + filled * soft).clip(0, 255).astype(np.uint8)
    del lama; torch.cuda.empty_cache()

BG_CLEAN = OUT / "background_clean.png"
Image.fromarray(bg).save(BG_CLEAN)
print(f"Background ready ({time.time() - t0:.0f} s). Saved to {BG_CLEAN}")
print("If the empty room doesn't look right, ask your image AI for 'the same room, empty' instead, "
      "untick BACKGROUND_HAS_ME, and run Step 0 and Step 3 again.")
show_images([BG_CLEAN], 240)

In [ ]:
#@title B4b · Cut you out of every frame with MatAnyone 2 (the slow part)
t0 = time.time()
from matanyone2 import MatAnyone2, InferenceCore
from matanyone2.utils.inference_utils import gen_dilate, gen_erosion

# 1) A mask of you on the first frame, so MatAnyone knows who to follow.
m0 = person_mask(frame0)
first_mask = ((m0 > 127) * 255).astype(np.uint8)
Image.fromarray(first_mask).save(WORK / "first_mask.png")
overlay = frame0.copy(); overlay[first_mask == 0] = (overlay[first_mask == 0] * 0.25).astype(np.uint8)
print("This is who will be cut out (everything dark is background):")
show_images([overlay], 200)
assert first_mask.mean() > 2, "No person found in the first frame. Try a START where you're clearly in view."
import gc; gc.collect()

# 2) Run MatAnyone 2 frame by frame (same steps as its own video function, but streamed,
#    so a 1-minute 1080p video fits in Colab's memory).
model = MatAnyone2.from_pretrained("PeiqingYang/MatAnyone2")
proc = InferenceCore(model, device=DEV)
N_WARMUP = 10
short = min(W, H)
if short > MATTING_SIZE:
    nh, nw = int(H / short * MATTING_SIZE), int(W / short * MATTING_SIZE)
else:
    nh, nw = H, W

mask = gen_erosion(gen_dilate(first_mask, 10, 10), 10, 10)
mask_t = torch.from_numpy(mask).float().to(DEV)
if (nh, nw) != (H, W):
    mask_t = F.interpolate(mask_t[None, None], size=(nh, nw), mode="nearest")[0, 0]

ALPHA = WORK / "alpha.mkv"
aw = writer(ALPHA, W, H, FPS, gray=True)
plate_sum = torch.zeros(3, H, W, device=DEV); plate_cnt = torch.zeros(1, H, W, device=DEV)

def frames_with_warmup():
    for _ in range(N_WARMUP):
        yield frame0
    yield from read_frames(segment, W, H)

done, t_loop = 0, time.time()
with torch.inference_mode(), torch.amp.autocast(device_type=DEV.type, enabled=DEV.type == "cuda"):
    for ti, fr in enumerate(frames_with_warmup()):
        full = torch.from_numpy(fr.copy()).to(DEV).permute(2, 0, 1).float()
        img = full if (nh, nw) == (H, W) else F.interpolate(full[None], size=(nh, nw), mode="area")[0]
        img = img / 255.0
        if ti == 0:
            proc.step(img, mask_t, objects=[1])
            prob = proc.step(img, first_frame_pred=True)
        elif ti <= N_WARMUP:
            prob = proc.step(img, first_frame_pred=True)
        else:
            prob = proc.step(img)
        if ti < N_WARMUP:
            continue
        a = proc.output_prob_to_mask(prob).float().clamp(0, 1)
        if a.shape[-2:] != (H, W):
            a = F.interpolate(a[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0]
        a = a[None]
        # Remember what your real room looks like where you're not standing (used to clean your edges).
        bgm = (a < 0.02).float()
        plate_sum += full / 255.0 * bgm; plate_cnt += bgm
        aw.stdin.write((a[0] * 255 + 0.5).byte().cpu().numpy().tobytes())
        done += 1
        if done % 150 == 0:
            rate = done / (time.time() - t_loop)
            print(f"  {done}/{N_FRAMES} frames  ({rate:.1f} frames/s, about {(N_FRAMES - done) / rate / 60:.1f} min left)")
aw.stdin.close(); aw.wait()

torch.save({"sum": plate_sum.cpu(), "cnt": plate_cnt.cpu()}, WORK / "plate.pt")
del model, proc; torch.cuda.empty_cache()
print(f"Cut-out done: {done} frames in {(time.time() - t0) / 60:.1f} min "
      f"({done / (time.time() - t_loop):.1f} frames/s).")

In [ ]:
#@title B5 · Put you in the new room (re-run after changing the realism sliders)
t0 = time.time()
with torch.inference_mode():
    # The new background, softly blurred like a real lens.
    Bn = torch.from_numpy(np.array(Image.open(BG_CLEAN).convert("RGB"))).to(DEV).permute(2, 0, 1).float() / 255
    Bn = gblur(Bn, BACKGROUND_BLUR * H / 1920 * 1.0)
    px = max(W, H) / 1920                                  # scale effects with the video size
    Bwrap = big_blur(Bn, 14 * px)

    # Your real room (from Step 4), filled in where you were always standing.
    pl = torch.load(WORK / "plate.pt")
    cnt = pl["cnt"].to(DEV); plate = pl["sum"].to(DEV) / cnt.clamp(min=1)
    valid = (cnt >= 3).float()
    fill_s = 40 * px
    filled = big_blur(plate * valid, fill_s) / big_blur(valid, fill_s).clamp(min=1e-3)
    plate = plate * valid + filled * (1 - valid)
    valid_soft = big_blur(valid, 3 * px)

    def to_ycc(x):
        y = 0.2126 * x[0] + 0.7152 * x[1] + 0.0722 * x[2]
        return torch.stack([y, (x[2] - y) / 1.8556, (x[0] - y) / 1.5748])
    def from_ycc(c):
        y, cb, cr = c
        r = y + 1.5748 * cr; b = y + 1.8556 * cb
        g = (y - 0.2126 * r - 0.0722 * b) / 0.7152
        return torch.stack([r, g, b])

    def foreground(I, a):
        """Your colours with the old room's colour removed from the edges."""
        solved = (I - (1 - a) * plate) / a.clamp(min=0.05)
        spread = gblur(I * a, 4 * px) / gblur(a, 4 * px).clamp(min=1e-4)
        w = ((a - 0.15) / 0.35).clamp(0, 1) * valid_soft
        Fg = w * solved + (1 - w) * spread
        Fg = torch.where(a > 0.97, I, Fg)
        return Fg.clamp(0, 1)

    # Colour match, worked out once (so it never flickers): nudge your average colour towards the room's.
    a0 = next(read_frames(ALPHA, W, H, gray=True))
    a0 = torch.from_numpy(a0.copy()).to(DEV).permute(2, 0, 1).float() / 255
    I0 = torch.from_numpy(frame0.copy()).to(DEV).permute(2, 0, 1).float() / 255
    fg_px = to_ycc(foreground(I0, a0))[:, a0[0] > 0.95]
    bg_ycc = to_ycc(Bn).flatten(1)
    if fg_px.shape[1] < 100:
        fg_px = to_ycc(I0).flatten(1)                      # fallback: whole first frame
    shift = (bg_ycc.mean(1) - fg_px.mean(1)) * COLOR_MATCH
    shift[0] *= 0.5                                        # brightness: only half, so your face stays readable
    shift = shift.view(3, 1, 1)

    # Grain: measure your camera's noise on the real room, add the same amount to the new one.
    hp = (I0 - gblur(I0, 1.0)).mean(0)
    bgpix = hp[(a0[0] < 0.02)]
    noise = float(1.4826 * bgpix.abs().median()) if bgpix.numel() > 1000 else 0.01
    noise = min(noise, 0.05) * GRAIN

    out_path = OUT / f"{src.stem}_new_background.mp4"
    vw = writer(out_path, W, H, FPS, audio_from=segment if seg["audio"] else None)
    n = 0
    for fr, af in zip(read_frames(segment, W, H), read_frames(ALPHA, W, H, gray=True)):
        I = torch.from_numpy(fr.copy()).to(DEV).permute(2, 0, 1).float() / 255
        a = torch.from_numpy(af.copy()).to(DEV).permute(2, 0, 1).float() / 255
        if EDGE_TIGHTEN > 0:
            a = ((a - EDGE_TIGHTEN) / (1 - EDGE_TIGHTEN)).clamp(0, 1)
        Fg = foreground(I, a)
        if COLOR_MATCH > 0:
            Fg = from_ycc(to_ycc(Fg) + shift).clamp(0, 1)
        bgf = Bn
        if noise > 0:
            bgf = (Bn + torch.randn(1, H, W, device=DEV) * noise).clamp(0, 1)
        comp = a * Fg + (1 - a) * bgf
        if LIGHT_WRAP > 0:
            wrap = (big_blur(1 - a, 10 * px) * a * LIGHT_WRAP).clamp(0, 1)
            comp = comp * (1 - wrap) + Bwrap * wrap
        vw.stdin.write((comp.clamp(0, 1) * 255 + 0.5).byte().permute(1, 2, 0).contiguous().cpu().numpy().tobytes())
        n += 1
        if n == 1:
            sample = (comp.clamp(0, 1) * 255).byte().permute(1, 2, 0).cpu().numpy()
    vw.stdin.close(); vw.wait()

# Side-by-side check video (original | new), smaller so it's quick to open.
compare = OUT / f"{src.stem}_compare.mp4"
ch = min(H, 960) // 2 * 2
run(["ffmpeg", "-v", "error", "-y", "-i", str(segment), "-i", str(out_path), "-filter_complex",
     f"[0:v]scale=-2:{ch}[a];[1:v]scale=-2:{ch}[b];[a][b]hstack[v]", "-map", "[v]", "-map", "0:a?",
     *X264, "-c:a", "aac", "-shortest", str(compare)])

fin = probe(out_path)
print(f"Done in {time.time() - t0:.0f} s.")
print(f"Your video: {out_path}")
print(f"Check video (original | new): {compare}")
print(f"{fin['w']}x{fin['h']}, {fin['fps']:.2f} fps, {fin['dur']:.1f} s, sound: {'yes' if fin['audio'] else 'no'}")
show_images([frame0, sample], 220)

In [ ]:
#@title B6 · Soft grade + save as the new edit.mp4 (what Claude Code uses)
t0 = time.time()
GRADE = ("hqdn3d=2:2:3:3,curves=all='0/0 0.25/0.22 0.6/0.61 1/0.98',eq=saturation=1.05,"
         "colorbalance=rs=0.02:bs=0.05:bm=0.02,unsharp=5:5:0.3,vignette=PI/7")
graded = WORK / "edit_swapped_graded.mp4"
run(["ffmpeg", "-v", "error", "-y", "-i", str(out_path), "-vf", GRADE + ",format=yuv420p", *X264, *TAGS,
     "-c:a", "copy", "-movflags", "+faststart", str(graded)])
res = Path(RES)
backup = res / "edit_original_background.mp4"
if not backup.exists():
    shutil.copy(res / "edit.mp4", backup)
a, b = probe(backup), probe(graded)
assert abs(a["dur"] - b["dur"]) < 0.1, f"Duration changed ({a['dur']:.2f} → {b['dur']:.2f} s): not replacing edit.mp4"
shutil.copy(graded, res / "edit.mp4")
short720 = "scale='if(gt(iw,ih),-2,720)':'if(gt(iw,ih),720,-2)'"
run(["ffmpeg", "-v", "error", "-y", "-i", str(graded), "-vf", short720, "-c:v", "libx264", "-preset", "veryfast",
     "-b:v", "1100k", "-maxrate", "1300k", "-bufsize", "2600k", "-c:a", "aac", "-b:a", "96k", "-movflags", "+faststart",
     str(res / "preview-720p.mp4")])
shutil.copy(compare, res / "compare-background.mp4")
eb = subprocess.run(["ffmpeg", "-hide_banner", "-i", str(res / "edit.mp4"), "-af", "ebur128=peak=true", "-f", "null", "-"],
                    capture_output=True, text=True).stderr
import re
lufs = re.findall(r"I:\s+(-?[\d.]+) LUFS", eb)[-1]
meta = json.load(open(res / "meta.json"))
meta.update({"graded": True, "backgroundSwapped": True, "backgroundImage": str(BACKGROUND_IMAGE), "lufs": float(lufs),
             "swapSettings": {"matting": MATTING_SIZE, "colorMatch": COLOR_MATCH, "lightWrap": LIGHT_WRAP,
                              "blur": BACKGROUND_BLUR, "grain": GRAIN, "edgeTighten": EDGE_TIGHTEN}})
json.dump(meta, open(res / "meta.json", "w"), indent=1)
with open(res / "report.md", "a", encoding="utf8") as f:
    f.write(f"\n## Background swap\n- Done: MatAnyone 2 at {MATTING_SIZE}, colour match {COLOR_MATCH}, light wrap {LIGHT_WRAP}, "
            f"blur {BACKGROUND_BLUR}, grain {GRAIN}; soft grade applied after the swap; {float(lufs):.1f} LUFS\n"
            f"- Original kept as `edit_original_background.mp4`; side-by-side check: `compare-background.mp4`\n")
print(f"Saved ({time.time() - t0:.0f} s). {res}/edit.mp4 now has the new background, graded, {float(lufs):.1f} LUFS.")
print("Watch preview-720p.mp4 / compare-background.mp4 on your phone, then give Claude Code the link of", res)

In [ ]:
#@title Optional · Watch the result here
import base64
p = Path(RES) / "compare-background.mp4"
if p.stat().st_size < 50e6:
    display(HTML(f'<video width=480 controls src="data:video/mp4;base64,{base64.b64encode(p.read_bytes()).decode()}"></video>'))
else:
    print("The file is large, so open it from your Drive:", p)